<a href="https://colab.research.google.com/github/icequeen101/Innovative-Applications-of-Machine-Learning-in-Aerospace-Industries/blob/main/Activity_1_1_Simple_Perceptron.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

<img src="https://drive.google.com/uc?id=1eBlhnvWo94RnzPK-86F5MzfFd86Cjb9C" width="65">

Created by: The AVELA Team


# **0. Welcome to Neural Network 101 <a style="text-decoration:none;" href="https://towardsdatascience.com/cheat-sheet-for-google-colab-63853778c093"><font color='blue'>G</font><font color='red'>o</font><font color='Goldenrod'>o</font><font color='blue'>g</font><font color='green'>l</font><font color='red'>e</font><font color="black"> Colab</font></a> notebook!**

Make sure to read <u>every</u> cell in this notebook to get your Python code working! \\
To run code, all you have to do is click the *run* button ▶️ (triangle in a circle, sometmes appears as brackets [ ]). \\

FIRST: Save a copy of this Google Colab notebook to your own drive by selecting: **File** -> **Save a copy in Drive**

Your job will be to **replace** the question marks (?) in each coding cell with the **correct** information explained in the comments. Then run the cell! \\

NOTE: If these activities feel difficult, try completing these [introduction to Pytorch & Python](https://tinyurl.com/AVELA-pytorch-intro) activities first.

### About This Neural Network 101 Colab Sheet
In this sheet we will learn how to create our own neural networks using PyTorch. This model will be less complicated, since it may be your first exposure to PyTorch. For more complex models, GPUs would make the training much faster.

- Learn to upload datasets to Google Colab
- Create and initialize class architectures
- Classify different articles of clothing
- Generalize this architecture for other applications

# **1. Downloading The Dataset**


Before we start to modify the perceptron algorithm, let's first download the dataset to our Google Colab and import other functions.

In [ ]:
# Start by importing some of our needed libraries
? ? # import the numpy library
? ? # import the torch library

In [ ]:
#@title  ↓Click "run" { display-mode: "form" }
# Import all necessary libraries for this Google Colab sheet
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
!pip install --upgrade gspread pandas
import pandas as pd
import gspread
from google.colab import auth
from google.auth import default
from sklearn.metrics import accuracy_score
from scipy.ndimage import gaussian_filter
# Authenticate and create the client
auth.authenticate_user()
creds, _ = default()
client = gspread.authorize(creds)
spreadsheet_id = '1RUhTd8FXVLcL99_PNwbHPU96qXqWlF18L8YSSXfn8Bk' # dataset hyperlink: https://docs.google.com/spreadsheets/d/1RUhTd8FXVLcL99_PNwbHPU96qXqWlF18L8YSSXfn8Bk/edit?usp=drive_link
sheet = client.open_by_key(spreadsheet_id).sheet1

def show_image_labels(img, label):
    plt.imshow(img.reshape(28, 28))
    plt.title(lbls[label])
    plt.show()

def dataframe_clean(df):
    # The below code sets the first row of the DataFrame (df) as the header
    df.columns = df.iloc[0]
    df = df[1:]
    # Convert the 'pixels' columns to numeric type
    for col in df.columns[:]:
      df[col] = pd.to_numeric(df[col], errors='coerce')  # invalid values will be set as NaN
    return df

def train(model, X_train, y_train, X_val, y_val, epochs=15, batch_size=32, lr=1e-3):
    """
    Q:  write the training loop following the schema shown above.

    Inputs
    - model: the model to be trained - a PyTorch nn.Module class object
    - X_train, y_train, X_val, y_val: training and validation data
    - epochs: num epochs, or the number of times we want to run through the entire training data
    - batch_size: number of data points per batch
    - lr: learning rate
    - optimizer: optimizer used

    Outputs
    - losses: a list of losses
    - accuracies: a list of validation accuracies
    - train_accs: a list of training accuracies
    """

    batches = len(X_train)//batch_size   # using batch_size, determine the number of batches needed

    loss_fn = nn.CrossEntropyLoss()                             # read the write-up for an explanation on CrossEntropyLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)     # read the write-up for an explanation on Adam

    losses = []
    train_accs = []
    accuracies = []

    for epoch in range(epochs):
        for i in range(0, len(X_train), batch_size):
            X_batch = X_train[i:i+batch_size]
            y_batch = y_train[i:i+batch_size]

            logits = model(X_batch)
            ypred= torch.argmax(logits, dim=1)

            loss = loss_fn(logits, y_batch)
            losses.append(loss)
            # these 3 functions will follow you whenever you train a model with PyTorch
            optimizer.zero_grad()   # erases the gradients from the previous epoch (sets all gradients to 0)
            loss.backward()         # calculates the gradients with respect to every single weight matrix in the model
            optimizer.step()        # takes ONE learning step with the gradients just calculated

        # feel free to use sklearn's accuracy_score function
        # calculate the training accuracy
        accuracy = accuracy_score(y_batch.detach().numpy(), ypred.detach().numpy())
        train_accs.append(accuracy)
        # calculate the validation accuracy and append the loss of this epoch
        logits_val = model(X_val)
        y_val_pred = torch.argmax(logits_val, dim=1)
        acc_val = accuracy_score(y_val.numpy(), y_val_pred.detach().numpy())
        accuracies.append(acc_val)

        # print epoch, loss, and current test accuracy
        print(f"Epoch {epoch}:\t Validation Accuracy {acc_val*100}% & Loss {loss}")

    return losses, accuracies, train_accs

def graph(accuracies, training_accs):
    """
    Q:  graph out the accuracies and training accuracies.
        make sure you label which curve is the validation/training accuracy.
        labels and titles are required.

    Inputs
    - accuracies: list of floats with length epochs
    - training_accs: list of floats with length epochs

    Outputs
    - None
    """
    print("The final epoch's validation accuracy is:", accuracies[-1]*100, "%")
    # Smooth the data using a Gaussian filter
    smoothed_accuracies = gaussian_filter(accuracies, sigma=2)
    smoothed_training_accs = gaussian_filter(training_accs, sigma=2)
    # Calculate averages
    avg_validation = numpy.mean(accuracies)*100
    avg_training = numpy.mean(training_accs)*100
    # Plot the data and add horizontal dashed lines for averages
    plt.plot(range(len(accuracies)), smoothed_accuracies*100, color='black', label= "Validation Data Accuracy")
    plt.axhline(avg_validation, color='black', linestyle='--', linewidth=1, label=f"Validation Avg: {avg_validation:.2f}%")
    plt.plot(range(len(training_accs)), smoothed_training_accs*100, color='red', label= "Training Data Accuracy")
    plt.axhline(avg_training, color='red', linestyle='--', linewidth=1, label=f"Training Avg: {avg_training:.2f}%")
    # Label the axes and add a legend and title
    plt.xlabel('Epoch')
    plt.ylabel("Accuracy (%)")
    plt.legend()
    plt.title("Model Accuracy vs. Epochs")
    plt.show()

## Setup & Helper Functions

This cell handles all the behind-the-scenes setup for the notebook. You don't need to modify anything here — just run it before proceeding.

---

### 1. Library Imports
Loads all necessary Python libraries including:
- **PyTorch** (`torch.nn`, `torch.nn.functional`) — for building and training neural networks
- **Matplotlib** — for plotting graphs and images
- **Pandas & gspread** — for loading and processing our dataset from Google Sheets
- **Scikit-learn** — for computing accuracy scores
- **SciPy** — for smoothing our accuracy curves when graphing

---

### 2. Google Sheets Authentication
Authenticates your Google account and connects to our shared dataset spreadsheet. The data is pulled directly from Google Sheets into the notebook — no file uploads needed.

---

### 3. Helper Functions

#### `show_image_labels(img, label)`
Displays a single image from the dataset alongside its label. Since each image is stored as a flat array of pixel values, it reshapes it into a **28×28 grid** before displaying.

#### `dataframe_clean(df)`
Cleans the raw data pulled from Google Sheets by:
- Promoting the first row to column headers
- Converting all pixel columns to numeric values (invalid entries become `NaN`)

#### `train(model, ...)`
The core training loop. For each epoch it:
1. Splits the training data into **mini-batches**
2. Runs each batch through the model to get predictions (forward pass)
3. Computes the **loss** using Cross-Entropy Loss
4. Backpropagates the error and updates the model weights using the **Adam optimizer**
5. Tracks and prints training and validation accuracy after each epoch

#### `graph(accuracies, training_accs)`
Plots validation vs. training accuracy over all epochs. A Gaussian smoothing filter is applied to the curves to reduce noise, and dotted horizontal lines mark the average accuracy for each.

In [ ]:
# Declare a variable named dataset and save our data to it
? = sheet.get_all_values()

# Convert the dataset variable to a DataFrame (df) using the pd.DataFrame() function
df = pd.DataFrame(?)

# Clean the DataFrame by passing df into the dataframe_clean() function
df = ?

# Display the first five rows of the df variable by calling the .head() function on it
?

In [ ]:
# Print the full size of the df variable using the .shape function
print(?) # This will print as (rows, columns)

# **2. Manipulating & Visualizing The Dataset**

In summary, you just imported our dataset, set it equal to the ```dataset``` variable, and converted the ```dataset``` variable into a DataFrame variable named ```df```. In the following coding block, we'll declare new variables ```X``` and ```Y```.

```X``` will get set equal to a [2D array](https://www.geeksforgeeks.org/python-arrays/) using the NumPy library (short for Numerical Python library). It should have a shape of (8000, 784) because our dataset has 8,000 different 28x28 images, totaling 784 pixels per image.


```Y``` will get set equal to a 1D array in a similar manner. It should have a shape of (8000,) because it will be set equal to the clothing *labels* for that image, but more on that later.

In [ ]:
# Declare a variable X and set it equal to the 2D NumPy array
? = df.drop(["label"], axis=1).to_numpy()

# Declare a variable Y and set it equal to the 1D NumPy array
? = df["label"].to_numpy()

# Print the shape of variable X
?

# Print the shape of variable Y
?

Before we visualize each of these images, let's first understand *why* we're using 28x28 images.

A **1080p** image is 1,920 pixels wide by 1,080 pixels high; which is a total of **2,073,600 pixels** per image.

To donwload and convert an image to a DataFrame in Google Colab, it would take much longer for image datasets that have (8000, 2073600) = **16,588,800,000 data points** compared to image datasets that only have (8000, 784) = **6,272,000 data points**.

To avoid needing more heavy computing hardware, pixel binning is commonly used to reduce the size of image data. By merging every $divisor1$ x $divisor2$ group of pixels into a single pixel, the size of a dataset can be greatly reduced.

For example, our dataset with 8,000 images that are 28x28 is **2,645 times smaller** than a dataset with 8,000 images that are 1080p!

<img src="https://drive.google.com/uc?id=1W7a8kx2ReFw4U1AO76ZRDJ_dndw9rkRw" width="600"> \\

In [ ]:
#@title If a 1008x1008 image has every 36x36 pixels binned into a single pixel, what is the final image resolution? (run this cell) {run: "auto"}

Image_resolution = 'Choose pixel density' #@param ["Choose pixel density", "1008x1008 resolution", "36x36 resolution", "28x28 resolution", "16x16 resolution"]

if (Image_resolution=="28x28 resolution"):
   print("Correct!")
else:
   print("Not quite right, try doing the math again!")

Each of the 8,000 images in our dataset are labeled as one of the following types of clothing:

```T-Shirt/Top, Trouser, Pullover, Dress, Coat, Sandal, Shirt, Sneaker, Bag, Boot```

These labels will help us train our model (the black box) to recognize features in the images (input) to determine a label type (output).

Next, we'll learn how to graph the dataset's images and their labels.

NOTE: Remember to tab in after declaring a function or loop.


In [ ]:
# Set a variable named lbls equal to a python list consiting of clothing labels above (make sure each item in the list is a string)
? = [?]

# Set a variable named num equal to the number of images and labels you want to print (e.g. 2, 3, 4, etc.)
?

# Create a for in range loop with a temporary variable named i that graphs the first num images and labels
? ? in range(?):
    ?(?, ?) # use the created function show_image_labels() function and pass in variables X[i], Y[i]

# **3. Implementing The Perceptron Algorithm**

In the perceptron algorithm, the forward function calculates the weighted sum of the input features and passes the result through an activation function to produce an output. This step models the perceptron's decision-making process for classification tasks.

Mathematically, it computes: $$y = f(w · x + b)$$

where:  
- $w$ are the weights,  
- $x$ are the inputs,  
- $b$ is the bias, and  
- $f$ is the activation function.


In [ ]:
#@title Select a forward function to train your model with (run this cell) {run: "auto"}

forward_func_sel = 'Choose forward function' #@param ["Choose forward function", "X = F.tanh(X)", "X = F.softmax(X)", "X = X # no activation", "X = F.relu(X)", "X = F.sigmoid(X)", "X = X * 0"]

class Perceptron(nn.Module):
    def __init__(self):
        super().__init__()
        input_dim =  X.shape[1]    # what should be the input dimension? what are we feeding the network?
        output_dim = len(numpy.unique(Y))     # what should be the output dimension? what do we want from the network after running model(), or also known as model.forward()?
        self.input_layer = nn.Linear(input_dim, output_dim)

    # Classify function to help with prediction
    def classify(self, X):
        """
        Q:  create a function classify that will take in X data points and produce
            the predicted classification of these points.

        Inputs
        - X: the torch.tensor matrix to be classified with shape (N, D)

        Outputs
        - labels: a torch.tensor with the shape (N, ), each item being X[i]'s
                  classification prediction
        """
        X = torch.tensor(X).type(torch.float32) # enforce smooth-running with the model
        logits = self(X)
        # Run softmax on the logits variable
        probabilities = torch.softmax(logits, dim=1)
        # Get the predicted label from the probability distribution
        labels = torch.argmax(probabilities, dim=1)
        return labels.type(torch.long)

    # Model(X) is equivalent as model.forward(X).
    # X is first passed into the input layer, then a non-linearity ReLU,
    # then the hidden layer, followed by a non-linearity ReLU. By the end,
    # the output_layer produces a logit matrix based on how many classes we want classified.
    def forward(self, X):
        X = self.input_layer(X)
        '''
        X = F.tanh(X) --- This wont produce the best results for output in the classify method
        X = F.softmax(X) --- This is the most appropriate activation function for the foward function, it will create a list of probabilities for each class
        X = no activation --- This is what we want
        X = F.sigmoid(X) --- This would be okay but will probably not give as good results as softmax. It is generally used for binary classifications
        X = F.relu(X) --- This is generally not used for an output for a classification class. it is generally used when you have larger models or a regression task
        X = X * num --- This is a wrong activation function to use, similar to doing no activtion
        X = X # no activation --- This activation function does nothing. in this use case won't get the best performing model
        '''
        if (forward_func_sel == "X = F.tanh(X)"):
          X = F.tanh(X)
        elif (forward_func_sel == "X = F.softmax(X)"):
          X = F.softmax(X, dim=1)
        elif (forward_func_sel == "X = F.relu(X)"):
          X = F.relu(X)
        elif (forward_func_sel == "X = F.sigmoid(X)"):
          X = F.sigmoid(X)
        elif (forward_func_sel == "X = X # no activation"):
          X = X
        else:
          X = X * 0 # for the "Choose forward function" and "X = X * 0" options
        logits = X
        return logits

Now, we'll need to transform our `X` and `Y` variables into Tensors using the `torch.tensor()` function. \
`X` should be type `torch.float32` and `Y` should be type `torch.long`.

In [ ]:
# Set the variable X to a tensor of type torch.float32
? = ?.?(X, dtype = ?)

# Set the variable Y to a tensor of type torch.long
? = ?.?(Y, dtype = ?)

Next, let's instantiate our model and verify the shape of its output.

It should have dimensions of (`num`, 10) once we input `num` samples and classify 1 of the 10 possible labels you declared above.

We can then take this output and apply a function to get the likelihood that each image corresponds to a label.

Lastly, you can print the perceptron algorithm's label prediction for the first `num` images in our dataset.

NOTE: Remember that the first value in a Python list has index 0. `lbls[0] = "T-Shirt/Top"`.

In [ ]:
# Declare a variable named model and set it equal to the output of the Perceptron() function call
?

# Declare a variable named out and set it equal to the output of the forward function you selected above
? = model(X[:num])

# Print the shape of the variable named out
?

# Declare a variable named predictions and set it equal to the predicted label for the first num images in the dataset
? = model.classify(X[:num])

# Print the predictions variable, which are the model's label predictions for the first num images in the dataset
?

# Create a for each loop with a temporary variable named i that iterates through the variable named predictions
?
  ? # print lbls[i]

The model's predictions don't seem quite right...

But let's verify that by graphing the model's predictions, and then comparing the output with our prior graphs.

NOTE: Remember that "T-shirt" and "shirt" are two different labels.

In [ ]:
# Create a for in range loop with a temporary variable named i that graphs the first num images and labels
?
  ? # use the show_image_labels() function and pass in variables X[i], predictions[i]

In [ ]:
#@title ↓run this cell to verify the actual labels for the first num images in the dataset { display-mode: "form" }
for i in range(num):
  print(lbls[df["label"].to_numpy()[i]])

The model might have been able to predict some of the labels correctly, but most of its predictions should be wrong.

This is because we haven't trained the model yet, the most important part of Deep Learning!

We'll need to write our own training loop to repeatedly train our model a set number of times (also called [epochs](https://deepai.org/machine-learning-glossary-and-terms/epoch)).


A typical training loop will have the following structure:

1.   Take the input *model*, *X training data*, *Y training data*, *X evaluation data*, and *Y evaluatioin data* ([training and evaluation data](https://youtu.be/dSCFk168vmo?feature=shared) **must** be different so the model is properly evaluated on data that it hasn't seen before)
2.   Take the input hyperparameters such as [learning rate, scheduler](https://wiki.cloudfactory.com/docs/mp-wiki/scheduler/overview-of-learning-rate-schedulers-in-ml#:~:text=It%20represents%20the%20size%20of,or%20based%20on%20performance%20improvements.), [batch size](https://www.sabrepc.com/blog/Deep-Learning-and-AI/Epochs-Batch-Size-Iterations?srsltid=AfmBOoocKYIiDDAnCW4B8D9bAQPWycwTCY0taS9ekIZ7b0O_uy3n56vV), [optimizers](https://www.analyticsvidhya.com/blog/2021/10/a-comprehensive-guide-on-deep-learning-optimizers/#h-what-is-optimizer), etc.
3. Initialize the model's [loss function](https://www.datacamp.com/tutorial/loss-function-in-machine-learning)
4. Complete an epoch, which iterates through the batches and keeps track of the losses and output metrics of choice (accuracy in our case).
5. Repeat step 3. for the input number of epochs

NOTE: If you selected the correct forward function above, you should get to between 70% - 80% accuracy below when training.



In [ ]:
# Declare a variable named model and set it equal to the Perceptron() function
?

# Start by training the model through 100 epochs
losses, accuracies, train_accs = train(model, X[0:7000], Y[0:7000], X[7000:8000], Y[7000:8000], epochs=?)

Typically, choosing the softmax() activation function would be correct because it is the most optimal for multiclass classifications. \
(e.g. our ten different clothing labels) \
However, the softmax() function was already applied in the classify function! \
So, by applying the softmax() activation in the forward function, we were effectively applying two activation functions on top of eachother. \
Mathmetically, this is equivalent to the composite function:
$$g(f(x))$$
or
$$y = g(f(w · x + b))$$

where:  
- $w$ are the weights,  
- $x$ are the inputs,  
- $b$ is the bias, and  
- $g$ is the classifier's activation function.
- $f$ is the forward activation function.

Softmax is typically used as the final activation in a neural network to normalize the outputs into a probability distribution over the classes. \\
(e.g. our classes are the ten different clothing labels we defined) \\
To properly configure the perceptron model, and avoid distoring our probaility distrubution by calling softmax() twice.

**We should have no activation** applied in the forward function. This will yield the best results!

# **4. Validate Your Model**

In [ ]:
# Use the graph() function and pass in the variables accuracies, train_accs
?

Now that we have trained a pretty decent model, we should save it for later use.

Let's save the model's parameters/weights and architecture.

In [ ]:
#@title ↓run this cell to save your model { display-mode: "form" }

# Save the model with the weights and architecture
torch.save(model, "simple_perceptron_complete.pth")
mod = torch.load("simple_perceptron_complete.pth")
mod.eval()
new_prediction =  mod.classify(X)

We can go back and check how accurate our now *trained* model predicts the first `num` labels in the dataset.

NOTE: You can increase the size of `num` to see more of your model's predictions!

In [ ]:
# Set a variable named num equal to the number of images and labels you want to check
?

# Create a for in range loop with a temporary variable named i that graphs the model's label predictions for the first num images
# Use the show_image_labels() function and pass in variables X[i], new_predictions[i]
?


In [ ]:
#@title ↓run this cell to verify the actual labels for the first num images in the dataset { display-mode: "form" }
for i in range(num):
  print(lbls[df["label"].to_numpy()[i]])